## Behavioral metrics.ipynb

| ipynb section | inputs | input structure | outputs | output structure | funcs used directly (src) | funcs via plotting helpers |
|---|---|---|---|---|---|---|
| Run all four sweeps | `BASE_PARAMS`, `{param}_grid`, `n_sims`, `collect_diagnostics`, `recency_k` | dict of scalars; list/1D array of floats; int; bool; int | `sweep_*` | dict keyed by param value → entry dict with guaranteed keys | `sweep_one_param` (`sweep.py`) | N/A |
| Recall accuracy | `recall_sims`, `N` | 2D array `(max_recalls, n_trials)`; int | mean unique-recall fraction per condition | scalar per condition (often assembled into 1D array over param grid) | `recall_accuracy(..., unique=True)` (`metrics.py`) | `plot_recall_accuracy` (`visualization.py`) |
| SPC | `SPC` (from sweep entry) or `recall_sims`, `N` | `SPC`: 1D length `N`; `recall_sims`: 2D; `N`: int | SPC curve per condition | 1D array length `N` | `compute_spc` (`metrics.py`) *(computed inside `sweep_one_param`)* | `plot_spc_sweep` (`visualization.py`) |
| PFR | `PFR` (from sweep entry) or `recall_sims`, `N` | `PFR`: 1D length `N`; `recall_sims`: 2D; `N`: int | PFR curve per condition | 1D array length `N` | `compute_pfr` (`metrics.py`) *(computed inside `sweep_one_param`)* | `plot_pfr_heatmap` (`visualization.py`) |
| Lag-CRP | `recall_sims`, `N` *(often recomputed via `lag_crp_with_counts` even if `lag_vals/lag_probs` are stored)* | 2D array; int | lag-CRP curve (and/or opportunities) | `compute_lag_crp`: `(lag_vals, lag_probs)` 1D arrays; `lag_crp_with_counts`: `(lags, crp, num, den)` 1D arrays | `compute_lag_crp` (`metrics.py`), `lag_crp_with_counts` (`metrics.py`) | `plot_lag_crp_sweep` (`visualization.py`) |
| Conditional lag rates | `recall_sims` → `(lags, num, den)`; threshold `k` | `lags/num/den`: aligned 1D arrays; `k`: int | conditional +1 / −1 rates; conditional large-lag summaries | 2 scalars per direction per condition (often assembled into 1D arrays over param grid) | `lag_crp_with_counts` (`metrics.py`), `conditional_forward_lag_rates(..., large_lag_thresh=k)` (`metrics.py`), `conditional_backward_lag_rates(..., large_lag_thresh=k)` (`metrics.py`) | `plot_directional_lag_rates` (`visualization.py`) |
| Unconditional lag summaries | `recall_sims`; threshold `k` | 2D array; int | unconditional ℓ=1 and ℓ≥k summaries | 2 scalars per condition (often assembled into 1D arrays over param grid) | `unconditional_transition_summaries(..., large_lag_thresh=k)` (`metrics.py`) | `plot_unconditional_lag_summaries` (`visualization.py`) |
| Conditional abs lag summaries | `recall_sims` → `(lags, num, den)`; threshold `k` | `lags/num/den`: aligned 1D arrays; `k`: int | conditional ℓ=1 and ℓ≥k summaries | 2 scalars per condition (often assembled into 1D arrays over param grid) | `lag_crp_with_counts` (`metrics.py`), `conditional_abs_lag_summaries(..., large_lag_thresh=k)` (`metrics.py`) | custom notebook plotting (no dedicated helper in `visualization.py` for this panel) |
| Lag-CRP diagnostics | `recall_sims`, `N` | 2D array; int | CRP + numerators + denominators (opportunities) | `lags/crp/num/den`: aligned 1D arrays | `lag_crp_with_counts` (`metrics.py`) | `plot_lag_crp_diagnostics` (`visualization.py`) |


### SPC and PFR

- SPC/PFR can be obtained in two equivalent ways:
  1. Directly from the sweep entry (computed inside `sweep_one_param`):  
     - `entry["SPC"]`, `entry["PFR"]`
  2. Recompute from raw simulation output (start from `recall_sims`):  
     - `compute_spc(entry["recall_sims"], N)`  
     - `compute_pfr(entry["recall_sims"], N)`

- In the pipeline, most plots read the stored arrays (`SPC`/`PFR`) from the sweep entry, but recomputing is used for validation or when only `recall_sims` is available.



### Lag-CRP recomputed even if lag_vals/lag_probs exist

- `sweep_one_param` stores:
  - `lag_vals, lag_probs = compute_lag_crp(recall_sims, N)`
  - and saves them into the sweep entry as `entry["lag_vals"]`, `entry["lag_probs"]`.

- Many analyses/plots still call: `lags, crp, num, den = lag_crp_with_counts(recall_sims, N)`
  - because it returns **additional** arrays needed downstream for conditional metrics:
  - `num`: observed transition counts per lag (numerators)
  - `den`: feasible transition counts per lag (opportunities / denominators)


### Observed transitions

- Observed transitions are defined the same way across **lag-based metrics**:
  - For each trial `t`, take the cleaned recall sequence:
    - `recalls = recall_sims[:, t][recall_sims[:, t] != 0]`
  - Consecutive recalls define observed transitions:
    - `(prev, next) = (recalls[r-1], recalls[r])`
    - observed lag: `next - prev`

### Opportunity counts
- Opportunity correction in `lag_crp_with_counts` adds:
  - At each recall step, it tracks which items have already been recalled, and define the feasible next choices as the remaining unrecalled items.
    - For the current `prev`, each feasible candidate `cand` implies a possible lag `cand - prev`.
    - increment `den[lag]` for each such feasible lag (opportunities),
    - increments `num[lag]` only for the actually observed `next`.

---

## `metrics.py`

| Function | Returns | Inputs (with defaults) | Opportunity-corrected? |
|---|---|---|---|
| `compute_spc` | P(recall) per serial position | `recall_sims, N` | N/A |
| `compute_pfr` | P(first recall) per serial position | `recall_sims, N` | N/A |
| `recall_accuracy` | Mean fraction of items recalled (unique by default) | `recall_sims, N, unique=True` | N/A |
| `compute_lag_crp` | CRP curve (includes lag 0) | `recall_sims, N` | **Yes** |
| `lag_crp_with_counts` | `lags`, `crp`, `num`, `den` (excludes lag 0) | `recall_sims, N` | **Yes** |
| `conditional_forward_lag_rates` | P(ℓ=+1), P(ℓ≥k) | `lags, num, den, large_lag_thresh=4` | **Yes** |
| `conditional_backward_lag_rates` | P(ℓ=−1), P(ℓ≤−k) | `lags, num, den, large_lag_thresh=4` | **Yes** |
| `conditional_abs_lag_summaries` | P(ℓ=1), P(ℓ≥k) | `lags, num, den, large_lag_thresh=4` | **Yes** |
| `unconditional_transition_summaries` | P(ℓ=1), P(ℓ≥k) | `recall_sims, large_lag_thresh=4` | **No** |

---
## Primary recall metrics



### `compute_spc(recall_sims, N)`

- Initialize `spc` as length-`N` array of zeros
- For each serial position `j` in `1..N`:
  - For each trial `t`:
    - Check whether `j` appears **anywhere** in that trial’s recall sequence:
      - `recalled_j[t] = any(recall_sims[:, t] == j)`
  - Set `spc[j] = mean(recalled_j)` across trials
- Return `spc`
---

### `compute_pfr(recall_sims, N)`

- Initialize `pfr` as length-`N` array of zeros
- Extract first recall for each trial:
  - `first = recall_sims[0, :]`
- Keep only trials where `first[t] != 0` (ignore “no recall” trials)
- For each serial position `j` in `1..N`:
  - Compute fraction of valid trials whose first recall equals `j`:
    - `pfr[j] = mean(first_valid == j)`
- Return `pfr`

---

### `recall_accuracy(recall_sims, N, unique=True)`

- For each trial `t`:
  - Get recalled items excluding padding:
    - `recalls = recall_sims[:, t]` where `recall_sims[:, t] != 0`
  - If `unique=True`:
    - `recalls = unique(recalls)` (remove repeats)
  - Compute trial accuracy:
    - `acc[t] = len(recalls) / N`
- Return `mean(acc)` across trials

---


### `compute_lag_crp(recall_sims, N)`  *(wrapper logic)*

- Call `lag_crp_with_counts(recall_sims, N)` to get:
  - `lags, crp, num, den`
- Construct the “full” lag axis that includes lag 0:
  - `lag_vals = [-(N-1), ..., -1, 0, +1, ..., +(N-1)]`
- Fill `lag_probs` using `crp` for nonzero lags and `0` at lag 0
- Return `(lag_vals, lag_probs)`

### `lag_crp_with_counts(recall_sims, N)`  *(core opportunity-corrected engine)*

- Define the set of lags to track:
  - `lags = [-(N-1), ..., -1, +1, ..., +(N-1)]`  (exclude 0)
- Initialize arrays (length = number of lags):
  - `num[lag] = 0`  (observed transitions)
  - `den[lag] = 0`  (opportunities)
- For each trial `t`:
  - Clean the recall list:
    - `recalls = recall_sims[:, t]` filtered to `!= 0`
  - Optionally skip trials with < 2 recalls (no transitions)
  - Track which items have already been recalled:
    - `recalled_set = {recalls[0]}` after the first recall
  - For each transition step `r = 1..len(recalls)-1`:
    - `prev = recalls[r-1]`, `next = recalls[r]`
    - Define feasible next items:
      - `candidates = {1..N} \ recalled_set`
    - For each candidate `cand` in `candidates`:
      - `possible_lag = cand - prev`
      - If `possible_lag != 0`, increment `den[possible_lag] += 1`
    - Compute observed lag:
      - `obs_lag = next - prev`
      - If `obs_lag != 0`, increment `num[obs_lag] += 1`
    - Update recalled set:
      - `recalled_set.add(next)`
- Compute CRP for each lag:
  - `crp[lag] = num[lag] / den[lag]` when `den[lag] > 0`, else `NaN`
- Return `(lags, crp, num, den)`

### Difference

| | `compute_lag_crp` | `lag_crp_with_counts` |
|---|---|---|
| **Includes lag 0?** | Yes (always CRP=0 at lag 0) | No |
| **Returns num/den?** | No (only `lag_vals, crp`) | Yes (`lags, crp, num, den`) |
| **Use case** | Plotting the CRP curve | Input to all scalar summary functions |

`lag_crp_with_counts` is the workhorse: its `num` and `den` arrays are the inputs to every conditional summary function.

---

## Conditional directional lag summaries

- These functions take the `lags`, `num`, `den` arrays from `lag_crp_with_counts` and produce scalar summaries for *signed* lags.

- Both use the **pool-then-divide** pattern: sum all relevant numerators, sum all relevant denominators, single division.


### Forward: `conditional_forward_lag_rates`

| Metric | Formula | 
|---|---|
| $P(\ell = +1)$ | $\displaystyle\frac{\text{num}(+1)}{\text{den}(+1)}$ |
| $P(\ell \geq k)$ | $\displaystyle\frac{\sum_{\ell \geq k}\text{num}(\ell)}{\sum_{\ell \geq k}\text{den}(\ell)}$ | 

#### `conditional_forward_lag_rates(lags, num, den, large_lag_thresh=4)`

- Compute conditional probability of `lag = +1`:
  - `p_plus1 = num[+1] / den[+1]` (if `den[+1] > 0`, else `NaN`)
- Compute conditional probability of “large forward lag” (`lag >= large_lag_thresh`):
  - `num_large = sum(num[lag] for lag in lags if lag >= large_lag_thresh)`
  - `den_large = sum(den[lag] for lag in lags if lag >= large_lag_thresh)`
  - `p_large = num_large / den_large` (if `den_large > 0`, else `NaN`)
- Return `(p_plus1, p_large)`

### Backward: `conditional_backward_lag_rates`

| Metric | Formula | 
|---|---|
| $P(\ell = -1)$ | $\displaystyle\frac{\text{num}(-1)}{\text{den}(-1)}$ | 
| $P(\ell \leq -k)$ | $\displaystyle\frac{\sum_{\ell \leq -k}\text{num}(\ell)}{\sum_{\ell \leq -k}\text{den}(\ell)}$ | 


### `conditional_backward_lag_rates(lags, num, den, large_lag_thresh=4)`

- Compute conditional probability of `lag = -1`:
  - `p_minus1 = num[-1] / den[-1]` (if `den[-1] > 0`, else `NaN`)
- Compute conditional probability of “large backward lag” (`lag <= -large_lag_thresh`):
  - `num_large = sum(num[lag] for lag in lags if lag <= -large_lag_thresh)`
  - `den_large = sum(den[lag] for lag in lags if lag <= -large_lag_thresh)`
  - `p_large = num_large / den_large` (if `den_large > 0`, else `NaN`)
- Return `(p_minus1, p_large)`

---

## Unconditional transition summaries 


### `unconditional_transition_summaries(recall_sims, large_lag_thresh=4)`

- Initialize counters:
  - `n_transitions = 0`
  - `n_abs1 = 0`   (count of transitions with lag = ±1)
  - `n_large = 0`  (count of transitions with lag >= threshold or <= -threshold)
- For each trial `t`:
  - `recalls = recall_sims[:, t]` filtered to `!= 0`
  - For each transition step `r = 1..len(recalls)-1`:
    - `prev = recalls[r-1]`, `next = recalls[r]`
    - `lag = next - prev`
    - Increment `n_transitions += 1`
    - If `abs(lag) == 1`, increment `n_abs1 += 1`
    - If `abs(lag) >= large_lag_thresh`, increment `n_large += 1`
- Convert counts to probabilities:
  - `p_abs1 = n_abs1 / n_transitions` (if `n_transitions > 0`, else `NaN`)
  - `p_large = n_large / n_transitions` (if `n_transitions > 0`, else `NaN`)
- Return `(p_abs1, p_large)`

The denominator is always the total number of transitions; it does not account for which lags were *available*. If a parameter setting causes fewer items to be recalled (fewer transitions), the distribution of possible lags changes mechanically, which can bias these summaries.

---

## Conditional absolute-value lag summaries


This is the opportunity-corrected counterpart of `unconditional_transition_summaries`. It answers: **ignoring direction, how contiguous are transitions after correcting for what was available?**

| Metric | Subset $S$ | Formula |
|---|---|---|
| $P(\lvert\ell\rvert = 1 \mid \text{opp})$ | $\{d : \lvert\ell\rvert = 1\}$ | $\dfrac{\text{num}_{\text{abs}}(1)}{\text{den}_{\text{abs}}(1)}$ | 
| $P(\lvert\ell\rvert \geq k \mid \text{opp})$ | $\{d : \lvert\ell\rvert \geq k\}$ | $\dfrac{\sum_{d \geq k}\text{num}_{\text{abs}}(d)}{\sum_{d \geq k}\text{den}_{\text{abs}}(d)}$ | 

### `conditional_abs_lag_summaries(lags, num, den, large_lag_thresh=4)`

- Compute conditional probability of `abs(lag) == 1`:
  - `num_abs1 = num[-1] + num[+1]`
  - `den_abs1 = den[-1] + den[+1]`
  - `p_abs1 = num_abs1 / den_abs1` (if `den_abs1 > 0`, else `NaN`)
- Compute conditional probability of “large absolute lag” (`abs(lag) >= large_lag_thresh`):
  - `num_large = sum(num[lag] for lag in lags if abs(lag) >= large_lag_thresh)`
  - `den_large = sum(den[lag] for lag in lags if abs(lag) >= large_lag_thresh)`
  - `p_large = num_large / den_large` (if `den_large > 0`, else `NaN`)
- Return `(p_abs1, p_large)`